**Staff and Contract**

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name)

df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Nr"                                : "employee_id",
    "Beschikbaarheid"                   : "availability_days",
    "kan ingepland worden op projecten:": "eligible_projects_raw",
    "Datum uitdienst (laatste dag)"     : "employment_end",
    "Startdatum contract"               : "contract_start",
    "Contract eind"                     : "contract_end",
    "Contractduur"                      : "contract_duration",
    "Contracturen"                      : "weekly_hours",
    "Functie"                           : "role",
    "Afdeling"                          : "department",
    "Bedrijf"                           : "company"
})

for col in ["employment_end", "contract_start", "contract_end"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

df["weekly_hours"] = pd.to_numeric(df["weekly_hours"], errors="coerce")
df["contract_duration"] = df["contract_duration"].replace({
    "Onbepaald": "Permanent", "Bepaald": "Fixed-term"})
df["role"] = df["role"].str.replace(r"^\d+\s*\|\s*", "", regex=True)

def get_worker_type(val):
    val = str(val).lower()
    if "stage" in val:    return "Intern"
    elif "oproep" in val: return "Freelancer"
    elif "bbl" in val:    return "Trainee"
    elif "vast" in val:   return "Permanent"
    return "Other"

df["worker_type"] = df["department"].apply(get_worker_type)

def parse_eligible(val):
    if pd.isna(val): return []
    val = str(val).lower().strip()
    result = []
    if "kantoor" in val:
        result.append("KANTOOR")
        if "combiworld" in val: result.append("COMBIWORLD")
        if "mdt" in val:        result.append("MDT")
        return result
    if "cc" in val:         result.append("CC")
    if "bsc" in val:        result.append("BSC")
    if "combiworld" in val: result.append("COMBIWORLD")
    if "mdt" in val:        result.append("MDT")
    if "jogg" in val:       result.append("BSC")
    if "BSC" in result and "CC" not in result:
        result.append("CC")
    return list(set(result))

df["eligible_projects"] = df["eligible_projects_raw"].apply(parse_eligible)

def is_dreammaker(val):
    if pd.isna(val): return False
    return "dreammaker" in str(val).lower()

df["is_dreammaker"] = df["eligible_projects_raw"].apply(is_dreammaker)

def is_kantoor(val):
    if pd.isna(val): return False
    val = str(val).lower().strip()
    return val in ["kantoor", "kantoor administratie"]

df["is_kantoor"] = df["eligible_projects_raw"].apply(is_kantoor)

DAY_MAP = {
    "maandag":   "Monday",
    "dinsdag":   "Tuesday",
    "woensdag":  "Wednesday",
    "donderdag": "Thursday",
    "vrijdag":   "Friday"
}

def parse_available_days(val):
    if pd.isna(val) or str(val).strip() == "":
        return ["Monday","Tuesday","Wednesday","Thursday","Friday"]
    val = str(val).lower()
    if "niet" not in val:
        available = [english for dutch, english in DAY_MAP.items() if dutch in val]
        return available if available else ["Monday","Tuesday","Wednesday","Thursday","Friday"]
    restricted = []
    for dutch, english in DAY_MAP.items():
        if dutch in val:
            idx = val.index(dutch)
            if "niet" in val[max(0, idx-10):idx]:
                restricted.append(english)
    all_days = ["Monday","Tuesday","Wednesday","Thursday","Friday"]
    return [d for d in all_days if d not in restricted]

df["available_days"] = df["availability_days"].apply(parse_available_days)

staff = df[[
    "employee_id",
    "employment_end",
    "contract_start",
    "contract_end",
    "contract_duration",
    "weekly_hours",
    "role",
    "department",
    "worker_type",
    "company",
    "eligible_projects",
    "is_dreammaker",
    "is_kantoor",
    "available_days"
]].copy()

year_start = pd.Timestamp("2025-01-01")
staff = staff[
    (staff["employment_end"].isna()) | (staff["employment_end"] >= year_start)
]
staff = staff.sort_values("contract_start", ascending=False)
staff = staff.drop_duplicates(subset=["employee_id"], keep="first")
staff = staff.sort_values("employee_id").reset_index(drop=True)
staff["employee_id"] = pd.to_numeric(staff["employee_id"], errors="coerce").astype("Int64")

# Save staff template (for company to use in app)
staff_export = staff.copy()
staff_export["eligible_projects"] = staff_export["eligible_projects"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x)
staff_export["available_days"] = staff_export["available_days"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x)
staff_export.to_excel("staff_template.xlsx", index=False)
print("staff_template.xlsx saved")

print(f"\nTotal employees active in 2025: {len(staff)}")
print(f"Dreammakers : {staff[staff['is_dreammaker']]['employee_id'].tolist()}")
print(f"Kantoor     : {staff[staff['is_kantoor']]['employee_id'].tolist()}")
print()
print(staff[["employee_id","role","worker_type","weekly_hours","company",
             "eligible_projects","is_dreammaker","is_kantoor"]].to_string())

Saving medewerkers cb cc (1).xlsx to medewerkers cb cc (1).xlsx
staff_template.xlsx saved

Total employees active in 2025: 78
Dreammakers : [36, 38, 42, 45, 46, 47, 48, 51, 58, 72]
Kantoor     : [32, 34, 80, 81, 82, 87, 100]

    employee_id                  role worker_type  weekly_hours              company           eligible_projects  is_dreammaker  is_kantoor
0            18   Combi. funct. sport   Permanent            36      CombiCoach B.V.                   [BSC, CC]          False       False
1            19        Buurtsp. coach       Other            40  Stichting Combibrug  [KANTOOR, COMBIWORLD, MDT]          False       False
2            23   Combi. funct. sport   Permanent            36      CombiCoach B.V.                   [BSC, CC]          False       False
3            26        Buurtsp. coach   Permanent            36  Stichting Combibrug  [KANTOOR, COMBIWORLD, MDT]          False       False
4            27            Combicoach   Permanent            34      Combi

**Hourly cost rate**

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df_raw = pd.read_excel(file_name, header=None)

# Monthly base rate columns
rate_col_indices = [2, 6, 10, 14, 18, 22, 27, 31, 35, 39, 43, 47]
month_names = ["jan","feb","mar","apr","may","jun",
               "jul","aug","sep","oct","nov","dec"]

df_costs = df_raw.iloc[2:, [0] + rate_col_indices].copy()
df_costs.columns = ["employee_id"] + month_names
df_costs = df_costs.dropna(subset=["employee_id"])

df_costs[month_names] = df_costs[month_names].replace({"-": np.nan, 0: np.nan})
for col in month_names:
    df_costs[col] = pd.to_numeric(df_costs[col], errors="coerce")

df_costs["avg_hourly_rate"] = df_costs[month_names].mean(axis=1)
df_costs = df_costs[["employee_id", "avg_hourly_rate"]].copy()
df_costs["employee_id"] = pd.to_numeric(
    df_costs["employee_id"], errors="coerce").astype("Int64")
df_costs = df_costs.dropna(subset=["avg_hourly_rate"])
df_costs = df_costs.drop_duplicates(subset=["employee_id"], keep="last")
df_costs = df_costs[
    df_costs["employee_id"].isin(staff["employee_id"])
].reset_index(drop=True)

# Fill missing rates
DREAMMAKER_RATE = 40
DEFAULT_RATE    = 50

staff_with_rate = staff.merge(df_costs, on="employee_id", how="left")

def fill_rate(row):
    if pd.notna(row["avg_hourly_rate"]): return row["avg_hourly_rate"]
    if row["is_dreammaker"]:             return DREAMMAKER_RATE
    return DEFAULT_RATE

staff_with_rate["avg_hourly_rate"] = staff_with_rate.apply(fill_rate, axis=1)

hourly_rate = dict(zip(
    staff_with_rate["employee_id"],
    staff_with_rate["avg_hourly_rate"]
))

print(f"Employees with rate from file : {len(df_costs)}")
print(f"Employees filled with default : {len(staff) - len(df_costs)}")
print(f"  - Dreammaker rate (€{DREAMMAKER_RATE}): {staff['is_dreammaker'].sum()}")
print(f"  - Default rate (€{DEFAULT_RATE})      : {len(staff) - len(df_costs) - staff['is_dreammaker'].sum()}")
print()
print("=== Hourly rate per employee ===")
print(staff_with_rate[["employee_id","role","is_dreammaker","avg_hourly_rate"]].to_string(index=False))

Saving 2025  uurprijs uva.xlsx to 2025  uurprijs uva.xlsx
Employees with rate from file : 34
Employees filled with default : 44
  - Dreammaker rate (€40): 10
  - Default rate (€50)      : 34

=== Hourly rate per employee ===
 employee_id                 role  is_dreammaker  avg_hourly_rate
          18  Combi. funct. sport          False        31.680833
          19       Buurtsp. coach          False        32.983333
          23  Combi. funct. sport          False        28.625000
          26       Buurtsp. coach          False        33.400833
          27           Combicoach          False        27.179167
          30       Buurtsp. coach          False        28.449000
          31       Buurtsp. coach          False        34.932500
          32                  DGA          False        50.000000
          33   Jongerenbegeleider          False        29.562500
          34  Mdw fin. administr.          False        35.619167
          35   Jongerenbegeleider          False 

/tmp/ipykernel_2242/622800085.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_costs[month_names] = df_costs[month_names].replace({"-": np.nan, 0: np.nan})
